# Lecture 3: Control Algorithms
This notebook addresses the problem of learning an **optimal policy** through interaction. Unlike Chapter 2, where the policy was fixed and we estimated its value, here both the policy and the value function are updated simultaneously.

---
**Topics:**
1. Multi-Armed Bandits and the Exploration-Exploitation Trade-off (UCB)
2. Q-Learning (Tabular)
3. SARSA (On-Policy TD Control)
4. Actor-Critic Methods
5. Policy Gradient (REINFORCE)


## 3.1 Setup and Helper Classes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

class FiniteMDP:
    """Finite MDP simulator for Chapter 3."""
    def __init__(self, states, actions, transitions, rewards, gamma=0.95):
        self.states       = states
        self.actions      = actions
        self.transitions  = transitions
        self.rewards      = rewards
        self.gamma        = gamma
        self.n_states     = len(states)
        self.n_actions    = len(actions)
        self.state_idx    = {s: i for i, s in enumerate(states)}
        self.action_idx   = {a: i for i, a in enumerate(actions)}
        self.current_state = states[0]

    def reset(self, state=None):
        self.current_state = state if state else np.random.choice(self.states)
        return self.current_state

    def step(self, action):
        s = self.current_state
        if (s, action) not in self.transitions:
            return s, 0.0, False
        next_states = list(self.transitions[(s, action)].keys())
        probs       = list(self.transitions[(s, action)].values())
        next_state  = np.random.choice(next_states, p=probs)
        reward      = self.rewards.get((s, action), 0.0)
        self.current_state = next_state
        done = (next_state == self.states[-1])
        return next_state, reward, done

    def true_q_star(self, tol=1e-9, max_iter=2000):
        """Compute Q* via value iteration."""
        Q = np.zeros((self.n_states, self.n_actions))
        for _ in range(max_iter):
            Q_new = np.zeros_like(Q)
            for i, s in enumerate(self.states):
                for j, a in enumerate(self.actions):
                    if (s, a) not in self.transitions:
                        continue
                    q = self.rewards.get((s, a), 0)
                    for s_next, prob in self.transitions[(s, a)].items():
                        k = self.state_idx[s_next]
                        q += self.gamma * prob * np.max(Q[k])
                    Q_new[i, j] = q
            if np.max(np.abs(Q_new - Q)) < tol:
                break
            Q = Q_new
        return Q

def build_gridworld(n=4, gamma=0.95):
    """Construct a 4x4 GridWorld MDP."""
    states  = list(range(n * n))
    actions = ['up', 'down', 'left', 'right']
    goal, trap = n*n - 1, n*n - 5

    def move(s, a):
        r, c = s // n, s % n
        if a == 'up'    and r > 0:   return s - n
        if a == 'down'  and r < n-1: return s + n
        if a == 'left'  and c > 0:   return s - 1
        if a == 'right' and c < n-1: return s + 1
        return s

    transitions, rewards = {}, {}
    for s in states:
        for a in actions:
            if s in [goal, trap]:
                transitions[(s, a)] = {s: 1.0}
                rewards[(s, a)] = 0.0
                continue
            ns_main = move(s, a)
            trans   = {ns_main: 0.7}
            for oa in actions:
                if oa != a:
                    ns_s = move(s, oa)
                    trans[ns_s] = trans.get(ns_s, 0) + 0.1
            transitions[(s, a)] = trans
            rewards[(s, a)]     = (1.0  if ns_main == goal else
                                   -1.0 if ns_main == trap else -0.01)
    return FiniteMDP(states, actions, transitions, rewards, gamma), goal, trap

env, GOAL, TRAP = build_gridworld()
Q_star = env.true_q_star()
V_star = np.max(Q_star, axis=1)
print(f"GridWorld ready. Q* shape: {Q_star.shape}")
print(f"Goal state: {GOAL}, Trap state: {TRAP}")

## 3.2 Multi-Armed Bandits: Exploration-Exploitation Trade-off

The multi-armed bandit problem is the canonical setting for studying **exploration vs exploitation**.

**Upper Confidence Bound (UCB1) — Szepesvári, pp. 38–40:**
$$A_t = \arg\max_a \left[ \hat{\mu}_a + \sqrt{\frac{2 \ln t}{N_t(a)}} \right]$$

- $\hat{\mu}_a$: empirical mean reward of arm $a$
- $N_t(a)$: number of times arm $a$ has been pulled
- Second term: exploration bonus (confidence width)


In [ ]:
class MultiArmedBandit:
    """K-armed stochastic bandit."""
    def __init__(self, means, stds=None):
        self.K = len(means)
        self.means = np.array(means)
        self.stds  = np.ones(self.K) if stds is None else np.array(stds)
        self.optimal_arm = np.argmax(self.means)

    def pull(self, arm):
        return np.random.normal(self.means[arm], self.stds[arm])

def epsilon_greedy(bandit, n_steps, epsilon):
    """Epsilon-greedy policy."""
    Q = np.zeros(bandit.K); N = np.zeros(bandit.K)
    rewards, regrets = [], []
    for t in range(n_steps):
        arm = np.random.randint(bandit.K) if np.random.rand() < epsilon else np.argmax(Q)
        r = bandit.pull(arm)
        N[arm] += 1; Q[arm] += (r - Q[arm]) / N[arm]
        rewards.append(r)
        regrets.append(bandit.means[bandit.optimal_arm] - bandit.means[arm])
    return np.array(rewards), np.cumsum(regrets)

def ucb1(bandit, n_steps, c=2.0):
    """UCB1 — Szepesvari Algorithm 5 (p. 39)."""
    Q = np.zeros(bandit.K); N = np.zeros(bandit.K)
    rewards, regrets = [], []
    for arm in range(bandit.K):          # pull each arm once
        r = bandit.pull(arm); N[arm] = 1; Q[arm] = r
        rewards.append(r)
        regrets.append(bandit.means[bandit.optimal_arm] - bandit.means[arm])
    for t in range(bandit.K, n_steps):
        arm = np.argmax(Q + np.sqrt(c * np.log(t + 1) / (N + 1e-9)))
        r = bandit.pull(arm); N[arm] += 1; Q[arm] += (r - Q[arm]) / N[arm]
        rewards.append(r)
        regrets.append(bandit.means[bandit.optimal_arm] - bandit.means[arm])
    return np.array(rewards), np.cumsum(regrets)

def thompson_sampling(bandit, n_steps):
    """Thompson Sampling (Bayesian approach)."""
    Q = np.zeros(bandit.K); N = np.zeros(bandit.K)
    rewards, regrets = [], []
    for t in range(n_steps):
        arm = np.argmax(np.random.normal(Q, 1.0 / (N + 1)))
        r = bandit.pull(arm); N[arm] += 1; Q[arm] += (r - Q[arm]) / N[arm]
        rewards.append(r)
        regrets.append(bandit.means[bandit.optimal_arm] - bandit.means[arm])
    return np.array(rewards), np.cumsum(regrets)

np.random.seed(42)
K, T, n_runs = 10, 2000, 30
true_means = np.random.normal(0, 1, K)
print(f"K={K}-armed bandit. Optimal arm: {np.argmax(true_means)} (\u03bc*={true_means.max():.3f})")

def run_bandit(algo_fn, n_runs=30):
    all_reg = []
    for _ in range(n_runs):
        _, cumreg = algo_fn(MultiArmedBandit(true_means), T)
        all_reg.append(cumreg)
    return np.mean(all_reg, axis=0), np.std(all_reg, axis=0)

reg_eps = run_bandit(lambda b,t: epsilon_greedy(b, t, 0.1))
reg_ucb = run_bandit(lambda b,t: ucb1(b, t, c=2.0))
reg_ts  = run_bandit(lambda b,t: thompson_sampling(b, t))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
t_ax = np.arange(T)
for (mean, std), label, color in [
    (reg_eps, '\u03b5-greedy (\u03b5=0.1)', 'red'),
    (reg_ucb, 'UCB1 (c=2)',             'steelblue'),
    (reg_ts,  'Thompson Sampling',       'green'),
]:
    axes[0].plot(t_ax, mean, color=color, linewidth=2, label=label)
    axes[0].fill_between(t_ax, mean-std, mean+std, color=color, alpha=0.15)

axes[0].set_xlabel('Step t'); axes[0].set_ylabel('Cumulative Regret')
axes[0].set_title(f'Bandit Algorithms Comparison\n({n_runs}-run average)', fontweight='bold')
axes[0].legend()

N_demo   = np.array([1, 1, 5, 10, 50, 100])
ucb_bonus = np.sqrt(2 * np.log(1000) / N_demo)
axes[1].bar(range(len(N_demo)), ucb_bonus, color='steelblue', alpha=0.8)
axes[1].set_xticks(range(len(N_demo)))
axes[1].set_xticklabels([f'N={n}' for n in N_demo])
axes[1].set_ylabel('UCB Bonus'); axes[1].set_xlabel('Times arm pulled (N)')
axes[1].set_title('UCB Exploration Bonus (t=1000)', fontweight='bold')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_bandit_en.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Final regret -> \u03b5-greedy: {reg_eps[0][-1]:.1f}  UCB1: {reg_ucb[0][-1]:.1f}  TS: {reg_ts[0][-1]:.1f}")

## 3.3 Q-Learning (Tabular)

### Algorithm 12 (Szepesvári, p. 47)

```
function QLearning(X, A, R, Y, Q):
    δ ← R + γ·max_{a'} Q[Y,a'] − Q[X,A]
    Q[X,A] ← Q[X,A] + α·δ
    return Q
```

**Update rule:**
$$Q_{t+1}(x, a) = Q_t(x, a) + \alpha_t \left[ R_{t+1} + \gamma \max_{a'} Q_t(X_{t+1}, a') - Q_t(x, a) \right]$$

Q-learning is **off-policy**: it converges to $Q^*$ regardless of the behaviour policy.

**Convergence guarantee:** When every state-action pair is visited infinitely often and Robbins-Monro conditions hold, $Q_t \to Q^*$ almost surely (Tsitsiklis 1994).


In [ ]:
def q_learning(env, n_episodes=3_000, alpha=0.1, epsilon=0.2,
               epsilon_decay=True, verbose=True):
    """
    Tabular Q-Learning — Algorithm 12 (Szepesvari)
    epsilon_decay: anneal epsilon over episodes
    """
    Q       = np.zeros((env.n_states, env.n_actions))
    _Q_star = env.true_q_star()
    eps_history, rmse_history = [], []

    for ep in range(n_episodes):
        state = env.reset(state=env.states[0])
        eps   = epsilon / (1 + ep * 0.001) if epsilon_decay else epsilon

        for _ in range(200):
            i = env.state_idx[state]
            action_idx = (np.random.randint(env.n_actions)
                          if np.random.rand() < eps else np.argmax(Q[i]))
            action = env.actions[action_idx]
            next_state, reward, done = env.step(action)

            j  = env.state_idx[next_state]
            td = reward + env.gamma * np.max(Q[j]) - Q[i, action_idx]
            Q[i, action_idx] += alpha * td

            state = next_state
            if done: break

        if ep % 50 == 0:
            rmse_history.append(np.sqrt(np.mean((Q - _Q_star) ** 2)))
            eps_history.append(eps)

    return Q, rmse_history, eps_history

print("Running Q-Learning...")
Q_est, rmse_hist, eps_hist = q_learning(env, n_episodes=5_000, alpha=0.1,
                                         epsilon=0.5, epsilon_decay=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im = axes[0].imshow(Q_est, aspect='auto', cmap='viridis')
axes[0].set_xlabel('Action index'); axes[0].set_ylabel('State index')
axes[0].set_title('Q-Learning: Q(s,a) Estimates', fontweight='bold')
axes[0].set_xticks(range(env.n_actions))
axes[0].set_xticklabels(env.actions, rotation=45)
plt.colorbar(im, ax=axes[0])

ep_ticks = np.arange(len(rmse_hist)) * 50
axes[1].plot(ep_ticks, rmse_hist, 'steelblue', linewidth=2)
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('RMSE(Q, Q*)')
axes[1].set_title('Q-Learning Convergence (Q \u2192 Q*)', fontweight='bold')
axes[1].fill_between(ep_ticks, rmse_hist, alpha=0.2)

n_grid    = 4
arrow_map = {'up': '\u2191', 'down': '\u2193', 'left': '\u2190', 'right': '\u2192'}
axes[2].set_xlim(-0.5, n_grid-0.5); axes[2].set_ylim(-0.5, n_grid-0.5)
axes[2].set_title('Learned Optimal Policy', fontweight='bold')
for s in env.states:
    i_s = env.state_idx[s]; row, col = s // n_grid, s % n_grid
    if s == GOAL:
        axes[2].text(col, n_grid-1-row, 'G', ha='center', va='center',
                    fontsize=16, color='green', fontweight='bold')
    elif s == TRAP:
        axes[2].text(col, n_grid-1-row, '\u2717', ha='center', va='center',
                    fontsize=16, color='red', fontweight='bold')
    else:
        axes[2].text(col, n_grid-1-row, arrow_map[env.actions[np.argmax(Q_est[i_s])]],
                    ha='center', va='center', fontsize=18)
axes[2].set_xticks(range(n_grid)); axes[2].set_yticks(range(n_grid))
axes[2].grid(True, alpha=0.5)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_qlearning_en.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Final RMSE: {rmse_hist[-1]:.5f}")

## 3.4 SARSA (On-Policy TD Control)

While Q-learning is **off-policy**, SARSA is **on-policy**: it learns the value of the policy it is actually executing.

**Update rule:**
$$Q_{t+1}(X_t, A_t) = Q_t(X_t, A_t) + \alpha \left[ R_{t+1} + \gamma Q_t(X_{t+1}, A_{t+1}) - Q_t(X_t, A_t) \right]$$

**Key difference:** SARSA uses the *actually selected* next action $A_{t+1}$, whereas Q-learning uses the *greedy* (max) action.


In [ ]:
def sarsa(env, n_episodes=3_000, alpha=0.1, epsilon=0.5, epsilon_decay=True):
    """SARSA — On-policy TD control."""
    Q     = np.zeros((env.n_states, env.n_actions))
    Q_opt = env.true_q_star()
    rmse_history = []

    def eps_greedy(state, eps):
        i = env.state_idx[state]
        return (np.random.randint(env.n_actions)
                if np.random.rand() < eps else np.argmax(Q[i]))

    for ep in range(n_episodes):
        eps   = epsilon / (1 + ep * 0.001) if epsilon_decay else epsilon
        state = env.reset(state=env.states[0])
        a_idx = eps_greedy(state, eps)

        for _ in range(200):
            action     = env.actions[a_idx]
            next_state, reward, done = env.step(action)
            next_a_idx = eps_greedy(next_state, eps)
            i  = env.state_idx[state]; j = env.state_idx[next_state]
            td = reward + env.gamma * Q[j, next_a_idx] - Q[i, a_idx]
            Q[i, a_idx] += alpha * td
            state, a_idx = next_state, next_a_idx
            if done: break

        if ep % 50 == 0:
            rmse_history.append(np.sqrt(np.mean((Q - Q_opt) ** 2)))

    return Q, rmse_history

print("Comparing Q-Learning vs SARSA...")
n_runs_ctrl = 10

def run_ctrl(algo_fn, n_runs=10):
    all_rmse = []
    for _ in range(n_runs):
        env2, _, _ = build_gridworld()
        if algo_fn == q_learning:
            _, rmse, _ = q_learning(env2, n_episodes=3_000, alpha=0.1,
                                     epsilon=0.5, verbose=False)
        else:
            _, rmse = sarsa(env2, n_episodes=3_000, alpha=0.1, epsilon=0.5)
        all_rmse.append(rmse)
    arr = np.array(all_rmse)
    return arr.mean(0), arr.std(0)

ql_mean, ql_std = run_ctrl(q_learning, n_runs=n_runs_ctrl)
sa_mean, sa_std = run_ctrl(sarsa,      n_runs=n_runs_ctrl)

fig, ax = plt.subplots(figsize=(10, 5))
ep_ticks = np.arange(len(ql_mean)) * 50
ax.plot(ep_ticks, ql_mean, 'steelblue',  linewidth=2.5, label='Q-Learning (off-policy)')
ax.fill_between(ep_ticks, ql_mean-ql_std, ql_mean+ql_std, color='steelblue', alpha=0.2)
ax.plot(ep_ticks, sa_mean, 'darkorange', linewidth=2.5, label='SARSA (on-policy)')
ax.fill_between(ep_ticks, sa_mean-sa_std, sa_mean+sa_std, color='darkorange', alpha=0.2)
ax.set_xlabel('Episode'); ax.set_ylabel('RMSE(Q, Q*)')
ax.set_title(f'Q-Learning vs SARSA ({n_runs_ctrl}-run average, shading=std)', fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_sarsa_vs_ql_en.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5 Actor-Critic Methods

The Actor-Critic architecture (Szepesvári, Section 3.4) maintains two components:

- **Actor:** Parameterises and improves the policy $\pi_\theta$.
- **Critic:** Estimates the value function $V_w$ and provides a learning signal.

### Updates (Sections 3.4.1–3.4.2, pp. 54–60)

**Critic (TD error):**
$$\delta_t = R_{t+1} + \gamma V_w(X_{t+1}) - V_w(X_t)$$
$$w \leftarrow w + \alpha_c \delta_t \nabla_w V_w(X_t)$$

**Actor (policy gradient):**
$$\theta \leftarrow \theta + \alpha_a \delta_t \nabla_\theta \log \pi_\theta(A_t \mid X_t)$$

**Softmax policy:**
$$\pi_\theta(a \mid s) = \frac{e^{\theta_{s,a}}}{\sum_{a'} e^{\theta_{s,a'}}}$$


In [ ]:
def actor_critic(env, n_episodes=4_000, alpha_actor=0.01, alpha_critic=0.1, gamma=0.95):
    """
    Tabular Actor-Critic — Szepesvari Section 3.4

    Actor  : theta[s, a]  policy parameters (softmax)
    Critic : w[s]         value function weights
    """
    _Q_ac   = env.true_q_star()
    _V_star = np.max(_Q_ac, axis=1)
    theta   = np.zeros((env.n_states, env.n_actions))
    w       = np.zeros(env.n_states)

    def softmax_policy(s):
        i = env.state_idx[s]
        prefs = theta[i] - np.max(theta[i])
        probs = np.exp(prefs); return probs / probs.sum()

    rewards_hist, rmse_hist = [], []

    for ep in range(n_episodes):
        state = env.reset(state=env.states[0])
        total_reward = 0

        for _ in range(300):
            probs      = softmax_policy(state)
            action_idx = np.random.choice(env.n_actions, p=probs)
            action     = env.actions[action_idx]
            next_state, reward, done = env.step(action)
            i = env.state_idx[state]; j = env.state_idx[next_state]

            delta = reward + gamma * w[j] * (1 - done) - w[i]
            w[i] += alpha_critic * delta

            grad_log_pi = -probs.copy(); grad_log_pi[action_idx] += 1.0
            theta[i]   += alpha_actor * delta * grad_log_pi

            total_reward += reward; state = next_state
            if done: break

        rewards_hist.append(total_reward)
        if ep % 50 == 0:
            rmse_hist.append(np.sqrt(np.mean((w - _V_star) ** 2)))

    return theta, w, rewards_hist, rmse_hist

print("Training Actor-Critic...")
theta_ac, w_ac, rew_hist, rmse_ac = actor_critic(env, n_episodes=5_000,
                                                    alpha_actor=0.02, alpha_critic=0.2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
n_grid    = 4
arrow_map = {'up': '\u2191', 'down': '\u2193', 'left': '\u2190', 'right': '\u2192'}
axes[0].set_title('Actor-Critic: Learned Policy', fontweight='bold')
axes[0].set_xlim(-0.5, n_grid-0.5); axes[0].set_ylim(-0.5, n_grid-0.5)

for s in env.states:
    i_s = env.state_idx[s]; row, col = s // n_grid, s % n_grid
    prefs = theta_ac[i_s] - np.max(theta_ac[i_s])
    probs = np.exp(prefs) / np.exp(prefs).sum()
    if s == GOAL:
        axes[0].text(col, n_grid-1-row, 'G', ha='center', va='center',
                    fontsize=16, color='green', fontweight='bold')
    elif s == TRAP:
        axes[0].text(col, n_grid-1-row, '\u2717', ha='center', va='center',
                    fontsize=16, color='red', fontweight='bold')
    else:
        axes[0].text(col, n_grid-1-row, arrow_map[env.actions[np.argmax(probs)]],
                    ha='center', va='center', fontsize=18)
axes[0].set_xticks(range(n_grid)); axes[0].set_yticks(range(n_grid)); axes[0].grid(True, alpha=0.5)

window = 100
smoothed = np.convolve(rew_hist, np.ones(window)/window, mode='valid')
axes[1].plot(rew_hist, alpha=0.3, color='steelblue', linewidth=0.8)
axes[1].plot(smoothed, 'steelblue', linewidth=2.5, label=f'Moving avg (w={window})')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Total Reward')
axes[1].set_title('Actor-Critic: Episode Rewards', fontweight='bold')
axes[1].legend()

x_states = np.arange(env.n_states)
axes[2].plot(x_states, V_star, 'k-', linewidth=2.5, label='V* (True)', zorder=5)
axes[2].plot(x_states, w_ac,   'r--', linewidth=2, label='Critic V\u0302')
axes[2].set_xlabel('State index'); axes[2].set_ylabel('Value')
axes[2].set_title('Critic: Value Function Estimate', fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_actor_critic_en.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 REINFORCE: Policy Gradient

REINFORCE (Williams 1992) estimates the policy gradient using Monte Carlo returns:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[ G_t \nabla_\theta \log \pi_\theta(A_t \mid X_t) \right]$$

**Update per step:**
$$\theta \leftarrow \theta + \alpha G_t \nabla_\theta \log \pi_\theta(A_t \mid X_t)$$

**Baseline:** Subtracting a state-dependent baseline $b(X_t)$ reduces variance without introducing bias (Szepesvári, Section 3.4.2):
$$\theta \leftarrow \theta + \alpha (G_t - b(X_t)) \nabla_\theta \log \pi_\theta(A_t \mid X_t)$$


In [ ]:
def reinforce(env, n_episodes=4_000, alpha=0.01, gamma=0.95, baseline=True):
    """
    REINFORCE Policy Gradient (Williams 1992)
    baseline=True: subtract a state-value baseline to reduce variance.
    """
    theta        = np.zeros((env.n_states, env.n_actions))
    baseline_val = np.zeros(env.n_states)

    def softmax_policy(s):
        i = env.state_idx[s]
        prefs = theta[i] - np.max(theta[i])
        probs = np.exp(prefs); return probs / probs.sum()

    def generate_episode_policy(max_steps=300):
        state = env.reset(state=env.states[0]); traj = []
        for _ in range(max_steps):
            probs = softmax_policy(state)
            a_idx = np.random.choice(env.n_actions, p=probs)
            next_s, r, done = env.step(env.actions[a_idx])
            traj.append((state, a_idx, r)); state = next_s
            if done: break
        return traj

    rewards_hist = []
    for ep in range(n_episodes):
        traj    = generate_episode_policy()
        T       = len(traj)
        returns = np.zeros(T); G = 0
        for t in range(T-1, -1, -1):
            G = traj[t][2] + gamma * G; returns[t] = G

        for t, (s, a_idx, r) in enumerate(traj):
            i = env.state_idx[s]; probs = softmax_policy(s)
            G_t = returns[t]
            if baseline:
                G_t -= baseline_val[i]
                baseline_val[i] += 0.01 * (returns[t] - baseline_val[i])
            grad = -probs.copy(); grad[a_idx] += 1.0
            theta[i] += alpha * G_t * grad

        rewards_hist.append(sum(r for _, _, r in traj))
    return theta, rewards_hist

print("Training REINFORCE (with and without baseline)...")
theta_rf,    rh_rf    = reinforce(env, n_episodes=3_000, alpha=0.02, baseline=False)
theta_rf_bl, rh_rf_bl = reinforce(env, n_episodes=3_000, alpha=0.02, baseline=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
window = 100
for rh, label, color in [
    (rh_rf,    'REINFORCE',            'darkorange'),
    (rh_rf_bl, 'REINFORCE + Baseline', 'steelblue'),
]:
    axes[0].plot(rh, alpha=0.2, color=color)
    axes[0].plot(np.convolve(rh, np.ones(window)/window, mode='valid'),
                 color=color, linewidth=2.5, label=label)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Total Reward')
axes[0].set_title('REINFORCE vs REINFORCE + Baseline', fontweight='bold')
axes[0].legend()

V_q  = np.max(Q_est, axis=1)
V_ac = w_ac
axes[1].plot(env.states, V_star, 'k-',  linewidth=3, label='V* (Optimal)', zorder=5)
axes[1].plot(env.states, V_q,    'b--', linewidth=2, label='Q-Learning V')
axes[1].plot(env.states, V_ac,   'r-.', linewidth=2, label='Actor-Critic V')
axes[1].set_xlabel('State index'); axes[1].set_ylabel('Value Estimate')
axes[1].set_title('Value Function Comparison Across Methods', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_reinforce_en.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.7 Comprehensive Multi-Run Comparison

In [ ]:
print("Running comprehensive comparison (may take a moment)...")
N_EPISODES = 2_000
N_RUNS     = 5

def avg_curve(algo_fn, n_runs=N_RUNS):
    all_curves = []
    for _ in range(n_runs):
        env_tmp, _, _ = build_gridworld()
        all_curves.append(algo_fn(env_tmp))
    min_len = min(len(c) for c in all_curves)
    arr = np.array([c[:min_len] for c in all_curves])
    return arr.mean(0), arr.std(0)

def ql_curve(env_tmp):
    _, rmse, _ = q_learning(env_tmp, n_episodes=N_EPISODES,
                             alpha=0.1, epsilon=0.5, verbose=False)
    return rmse

def sarsa_curve(env_tmp):
    _, rmse = sarsa(env_tmp, n_episodes=N_EPISODES, alpha=0.1, epsilon=0.5)
    return rmse

def ac_curve(env_tmp):
    _, _, _, rmse = actor_critic(env_tmp, n_episodes=N_EPISODES,
                                  alpha_actor=0.02, alpha_critic=0.2)
    return rmse

curves = {
    'Q-Learning':   avg_curve(ql_curve),
    'SARSA':        avg_curve(sarsa_curve),
    'Actor-Critic': avg_curve(ac_curve),
}

fig, ax = plt.subplots(figsize=(12, 6))
colors = {'Q-Learning': 'steelblue', 'SARSA': 'darkorange', 'Actor-Critic': 'green'}

for name, (mean, std) in curves.items():
    ep_ticks = np.arange(len(mean)) * 50
    ax.plot(ep_ticks, mean, color=colors[name], linewidth=2.5, label=name)
    ax.fill_between(ep_ticks, mean-std, mean+std, color=colors[name], alpha=0.15)

ax.set_xlabel('Episode', fontsize=13)
ax.set_ylabel('RMSE (Q or V \u2192 Optimal)', fontsize=13)
ax.set_title(f'Control Algorithms Comparison\n({N_RUNS}-run average, shading = std)',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch3_comparison_en.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n--- Final RMSE Summary ---")
for name, (mean, std) in curves.items():
    print(f"{name:<20}: {mean[-1]:.5f} \u00b1 {std[-1]:.5f}")

## 3.8 Summary

| Algorithm | On/Off-policy | Model | Convergence | Scalability |
|-----------|:------------:|-------|-------------|-------------|
| **Q-Learning**   | Off-policy | Model-free | $Q^*$ (global) | Tabular |
| **SARSA**        | On-policy  | Model-free | $Q^{\pi_\epsilon}$ | Tabular |
| **Actor-Critic** | On-policy  | Model-free | Local optimum | Scales with FA |
| **REINFORCE**    | On-policy  | Model-free | Local optimum | Scales with FA |
| **UCB**          | —          | Model-free | Regret minimisation | Bandit only |

### Key Concepts
- **Off-policy:** Behaviour policy ≠ target policy (Q-learning)
- **On-policy:** Behaviour policy = target policy (SARSA, Actor-Critic)
- **Exploration-Exploitation:** UCB achieves near-optimal exploration in bandits
- **Policy Gradient:** Optimises $\theta$ directly; value function serves as a baseline

### Practical Guidelines
1. Small, finite environments → **Q-Learning** or **SARSA**
2. Continuous state/action spaces → **Actor-Critic** + function approximation
3. Bandit problems → **UCB1** or **Thompson Sampling**
4. High-variance policy gradients → **REINFORCE + Baseline**
